# Milestone 1 lab, notebook 6 of 7: Why batch, and where it breaks

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and the source | Read the question. Meet the order system. Look at one order |
| 2. Read every order | Follow the pages of the API to the end |
| 3. The answer by hand | Work out the answer with plain Python. Meet the time zone trap |
| 4. Transform, then load | Put the orders in a warehouse table. Ask it one SQL question |
| 5. Every morning | Run it again the next day. Watch an old number change |
| **6. Why batch, and where it breaks** (you are here) | Measure how late the report is. Then watch the load fail |
| 7. The production version | The same logic as a tested program |

The pipeline runs once a morning. Is that good enough? This notebook measures how late the report is. Then it runs the pipeline for a few more mornings, until it breaks.

In [1]:
import pandas as pd
from lab_helpers import api, arden_time, bars, full_load, money, set_morning

`full_load` is the loop from notebook 2, kept in `lab_helpers.py`. It returns the orders, the number of calls, and whether the API refused.

## Step 1: How late was the refund?

It is Tuesday morning again. In notebook 5 you found order 1130, refunded on Monday.

In [2]:
set_morning("2026-06-09")
orders, calls, refused = full_load()

order = api.get("/orders/1130").json()
refunded = arden_time(order["refunded_at"])
shown = arden_time("2026-06-09T12:00:00Z")  # Tuesday 07:00 in Arden, the time of the run

print(
    "Refund given:           ", refunded.strftime("%A %H:%M"), f"({money(order['refunded_cents'])})"
)
print("Warehouse could show it:", shown.strftime("%A %H:%M"))
print("The refund waited", shown - refunded)

It is now Tuesday 2026-06-09 07:00:00 in Arden.
Refund given:            Monday 11:00 ($5.78)
Warehouse could show it: Tuesday 07:00
The refund waited 20:00:00


The refund happened on Monday at 11:00. The warehouse first showed it on Tuesday at 07:00. That wait has a name.

## Step 2: Two new words, latency and batch

| Word | Meaning | Here |
|---|---|---|
| **Latency** | "Latency is the time from a change in the source until the warehouse can show it." | 20 hours for this refund |
| **Batch** | "Batch means one run handles a whole slice of data, then stops. The slice is a time range." | One run a morning, for the day before |

Both definitions are from the D8Loop chapter [Batch vs Streaming vs Micro-Batch](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/batch-vs-streaming-vs-micro-batch).

One refund is one example. Measure the latency of every change the Tuesday run picked up.

In [3]:
monday_run = "2026-06-08T12:00:00Z"
waits = sorted(
    (shown - arden_time(o["updated_at"])).total_seconds() / 3600
    for o in orders
    if o["updated_at"] >= monday_run
)
print(f"{len(waits)} orders changed between the two runs.")
print(f"Shortest wait: {waits[0]:.1f} hours.   Longest wait: {waits[-1]:.1f} hours.")

29 orders changed between the two runs.
Shortest wait: 7.7 hours.   Longest wait: 21.6 hours.


A change just after a run waits almost a full day. The last change of the night still waits about eight hours, until the 07:00 run. With a daily batch, the latency is **up to one day**.

## Step 3: Is one day too slow?

Latency is a cost. The other side is what the reader needs. That has a word too: **freshness**, how new the data in the warehouse is.

Ask who reads the report. The manager reads it once, at 08:00. Nobody looks again until the next morning. Data from the 07:00 run is fresh enough.

### Compare batch with its opposites

| Shape | How it runs | Latency | Cost and effort |
|---|---|---|---|
| **Batch** | One run handles a whole slice, then stops. Here: once a morning | Up to one day | Lowest. A failed run is run again |
| **Micro-batch** | Batch with a very small slice. The job runs again and again, such as every minute | Up to one interval | Low to medium |
| **Streaming** | A job that never stops. It handles each event as it arrives | A short moment | Highest. You manage state, late events and restarts |

A faster pipeline would cost more and help no one here. The rule: **ask how fresh the result must be.** A daily report can wait until morning. A fraud alert cannot.

Streaming is not built in this project. You will run this pipeline as a micro-batch in milestone 5, to see where batch stops being enough.

## Step 4: What is still wrong

Batch is the right shape. The pipeline inside it is not right yet. Run it for four mornings, and measure.

In [4]:
measured = []
previous_run = None
for day in ["2026-06-08", "2026-06-09", "2026-06-10", "2026-06-11"]:
    api.post("/admin/clock", json={"morning": day})
    orders, calls, refused = full_load()
    run_time = api.get("/admin/clock").json()["now_utc"]
    new_or_changed = (
        None if previous_run is None else sum(o["updated_at"] >= previous_run for o in orders)
    )
    measured.append(
        {
            "morning": day,
            "orders_read": len(orders),
            "api_calls": calls,
            "new_or_changed": "" if refused or new_or_changed is None else new_or_changed,
            "result": "❌ refused: 429" if refused else "✅ loaded",
        }
    )
    previous_run = run_time

pd.DataFrame(measured)

,morning,orders_read,api_calls,new_or_changed,result
0,2026-06-08,226,10,,✅ loaded
1,2026-06-09,251,11,29,✅ loaded
2,2026-06-10,282,12,38,✅ loaded
3,2026-06-11,300,13,,❌ refused: 429


Read the table from top to bottom.

- Each morning the load reads **more** orders, because it reads every order Arden ever took.
- On Tuesday only 29 orders were new or changed. The load read all 251 anyway.
- The API allows 12 calls in 10 seconds. On Thursday the load needs 13. **Call 13 is refused**, and there is no report.

In [5]:
bars({row["morning"]: row["api_calls"] for row in measured}, unit=" calls")
print("The limit is 12 calls.")

2026-06-08  ███████████████████████████████ 10 calls
2026-06-09  ██████████████████████████████████ 11 calls
2026-06-10  █████████████████████████████████████ 12 calls
2026-06-11  ████████████████████████████████████████ 13 calls
The limit is 12 calls.


Nothing in the code changed between Wednesday and Thursday. The data grew, and the pipeline broke.

> **Still wrong:** it reloads every order every day. It is slow, and it hits the API's rate limit.

Milestone 2 fixes it. It loads only what changed.

## What you now know

- Latency is the time from a change in the source until the warehouse can show it. Here it is up to one day.
- This is a batch pipeline, and that is fresh enough for a report read once a morning.
- A full load grows every day. On the morning of 11 June the API refuses it.

**Say it in an interview:** "I chose batch because the report is read once a morning, so a latency of up to a day was fine and it was the cheapest thing to run. But my full load read every order every day, and it failed on the day it needed one call more than the API's rate limit allowed. That pushed me to an incremental load."

Next: [notebook 7](07_the_production_version.ipynb) shows the same logic as a tested program.